## Notebook Purpose

This notebook explores Google's [Places Api](https://developers.google.com/maps/documentation/places/web-service/overview?_gl=1*1h2my3u*_up*MQ..*_ga*MTQ1OTU0Mjc1NC4xNzY1MjMyMzIx*_ga_NRWSTWS78N*czE3NjUyMzIzMjEkbzEkZzEkdDE3NjUyMzIzNzkkajIkbDAkaDA.*_ga_SM8HXJ53K2*czE3NjUyMzIzMjEkbzEkZzAkdDE3NjUyMzIzMjEkajYwJGwwJGgw)

In [12]:
import requests
import os
import json

## Test Places Search requests for cafes in London

The [documentation](https://developers.google.com/maps/documentation/places/web-service/text-search?_gl=1*1is1mlh*_up*MQ..*_ga*MTE1NjAyMDMwNi4xNzY1Mjk4MzM2*_ga_NRWSTWS78N*czE3NjUyOTgzMzYkbzEkZzAkdDE3NjUyOTgzMzckajU5JGwwJGgw*_ga_SM8HXJ53K2*czE3NjUyOTgzMzYkbzEkZzAkdDE3NjUyOTgzMzYkajYwJGwwJGgw) was followed in order to create this test.

In [15]:
# Constants
places_url = 'https://places.googleapis.com/v1/places:searchText'
API_KEY = os.getenv('API_KEY')

In [16]:
headers = {
    'Content-Type': 'application/json',
    'X-Goog-API-Key': API_KEY,
    'X-Goog-FieldMask': '*',
    # IMPORTANT: using '*' for FieldMask returns all fields. I am only using this for the test - don't use when we know what fields we want returned because it could increase costs.
}

data = {
    'textQuery': 'cafes in Holborn, London, England'
}

response = requests.post(url=places_url, headers=headers, data=json.dumps(data))

if response.status_code == 200:
    print('Success!')
else:
    print(f'Error: {response.status_code}: {response.text}')

Success!


In [18]:
response.json()

{'places': [{'name': 'places/ChIJ8w5RD0obdkgRPwulvvc5VNw',
   'id': 'ChIJ8w5RD0obdkgRPwulvvc5VNw',
   'types': ['cafe', 'point_of_interest', 'food', 'establishment'],
   'nationalPhoneNumber': '020 7405 8565',
   'internationalPhoneNumber': '+44 20 7405 8565',
   'formattedAddress': '28 Red Lion St, London WC1R 4PS, UK',
   'addressComponents': [{'longText': '28',
     'shortText': '28',
     'types': ['street_number'],
     'languageCode': 'en-US'},
    {'longText': 'Red Lion Street',
     'shortText': 'Red Lion St',
     'types': ['route'],
     'languageCode': 'en'},
    {'longText': 'London',
     'shortText': 'London',
     'types': ['postal_town'],
     'languageCode': 'en'},
    {'longText': 'Greater London',
     'shortText': 'Greater London',
     'types': ['administrative_area_level_2', 'political'],
     'languageCode': 'en'},
    {'longText': 'England',
     'shortText': 'England',
     'types': ['administrative_area_level_1', 'political'],
     'languageCode': 'en'},
    {

## Test Places Aggregate API response

[This guide](https://developers.google.com/maps/documentation/places-aggregate/make-your-first-request?_gl=1*1kb9yk0*_up*MQ..*_ga*ODM0NTQyNDE5LjE3NjUyODM1NzE.*_ga_SM8HXJ53K2*czE3NjUyODM1NzEkbzEkZzAkdDE3NjUyODM1NzEkajYwJGwwJGgw*_ga_NRWSTWS78N*czE3NjUyODM1NzEkbzEkZzAkdDE3NjUyODM1NzEkajYwJGwwJGgw) shows how to set up a request.

This example aims to count how many cafes there are in the LSE postcode 'region'. 

In [8]:
# Constants
url = 'https://areainsights.googleapis.com/v1:computeInsights'
API_KEY = os.getenv('API_KEY')
place_id = 'places/ChIJWx2zXLUEdkgRNoC9_WcA2NE'
# Place ID for the LSE postcode (WC2A2AE), found using place ID Finder: https://developers.google.com/maps/documentation/javascript/examples/places-placeid-finder?_gl=1*1wj64qj*_up*MQ..*_ga*MTAwMjkzMjcwNi4xNzY1Mjg0NDM3*_ga_NRWSTWS78N*czE3NjUyODQ0MzckbzEkZzEkdDE3NjUyODYxNzAkajYwJGwwJGgw*_ga_SM8HXJ53K2*czE3NjUyODQ0MzckbzEkZzAkdDE3NjUyODYxMzYkajYwJGwwJGgw

In [9]:
headers = {
    'X-Goog-Api-Key': API_KEY,
    'Content-Type': 'application/json',
}

payload = {
    'insights': [
        'INSIGHT_COUNT'
    ],
    'filter':{
        'locationFilter': {
            'region': {
                'place': place_id
            }
        },
        'typeFilter':{
            'includedTypes':[
                'cafe'
            ]
        },
        'operatingStatus':[
            'OPERATING_STATUS_OPERATIONAL'
        ]
    }      
}

response = requests.post(url, headers=headers, json=payload)
print(response.status_code)

400


### Notes on API request

The code above adapts the guide provided by google (which is written using curl) to python. Where curl uses `--data`, I use `payload`.

In [10]:
# Inspect 403 response status code
try:
    error_data = response.json()
    print(json.dumps(error_data, indent=2))
except ValueError:
    print("Response is not JSON:")
    print(response.text)

{
  "error": {
    "code": 400,
    "message": "This region is not supported because it does not have a well defined geometry.",
    "status": "INVALID_ARGUMENT"
  }
}
